# task3-optimisation
Compare Q8_0 and Q4_K_M on the same ten prompts. Report aggregate decode speed and the largest resident model VRAM reading after a request. This excludes prompt processing from speed and is not a peak GPU-memory measurement.

Select Runtime → Change runtime type → hosted GPU, then Run all. Use a fresh runtime for each notebook. Save the downloaded results before disconnecting.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd

In [ ]:
import json
import subprocess
import time
from pathlib import Path
import requests

gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
print(gpu)
url = "http://127.0.0.1:11434/api"
if not Path("/usr/local/bin/ollama").exists():
    subprocess.run(["bash", "-o", "pipefail", "-c", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
try:
    requests.get(url + "/version", timeout=2).raise_for_status()
except requests.RequestException:
    server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        try:
            if requests.get(url + "/version", timeout=2).ok:
                break
        except requests.RequestException:
            continue
    else:
        raise RuntimeError(Path("ollama.log").read_text()[-2000:])

prompts = [
    ("Write just the sum of 17 and 28.", "45"),
    ("Alphabetise these words: kite, apple, fern.", "apple, fern, kite"),
    ("Return only JSON with city set to Pune and seats set to the integer 2.", "Valid object with both fields and the correct types"),
    ("A train leaves at 09:45 and travels for 50 minutes. Give the arrival time.", "10:35"),
    ("Extract just the order ID: Your order ORD-709 left today.", "ORD-709"),
    ("Rewrite 'The cat chased the mouse' in the passive voice.", "The mouse was chased by the cat."),
    ("List the prime numbers between 10 and 20.", "11, 13, 17, 19"),
    ("Asha has three pens. Ravi has two fewer pens than Asha. How many do they have together?", "4"),
    ("Summarise in one sentence: The office closes on Tuesday for repairs. Staff return Wednesday.", "Mentions closure for repairs and return Wednesday"),
    ("A note reads: The box is blue. What does the note say about its weight?", "Weight is unspecified"),
]
def generate(model, prompt):
    response = requests.post(url + "/generate", json={"model": model, "prompt": prompt, "stream": False,
        "keep_alive": "5m", "options": {"temperature": 0, "seed": 42, "num_ctx": 2048, "num_predict": 64}}, timeout=240)
    response.raise_for_status()
    return response.json()

result = {"gpu": gpu, "version": requests.get(url + "/version").json(), "runs": {}}
for quant in ("q8_0", "q4_K_M"):
    model = "qwen2.5:0.5b-instruct-" + quant
    subprocess.run(["ollama", "pull", model], check=True)
    generate(model, "Say hello.")
    rows = []
    for prompt, expected in prompts:
        reply = generate(model, prompt)
        loaded = next(m for m in requests.get(url + "/ps").json()["models"] if m["name"] == model)
        assert loaded["size_vram"] > 0, "Ollama is not using the GPU. Check the hosted runtime."
        rows.append({"prompt": prompt, "expected": expected, "answer": reply["response"],
                     "tokens": reply["eval_count"], "decode_ns": reply["eval_duration"],
                     "resident_vram_bytes": loaded["size_vram"]})
    speed = sum(r["tokens"] for r in rows) * 1e9 / sum(r["decode_ns"] for r in rows)
    memory = max(r["resident_vram_bytes"] for r in rows) / 2**20
    result["runs"][quant] = {"decode_tokens_per_second": speed, "resident_model_MiB": memory, "outputs": rows}
    print(quant, speed, "tokens/s;", memory, "MiB resident")
    subprocess.run(["ollama", "stop", model], check=True)
for left, right in zip(*(r["outputs"] for r in result["runs"].values())):
    print(left["prompt"], "Expected:", left["expected"], "Q8:", left["answer"], "Q4:", right["answer"], sep="\n")
Path("task3-results.json").write_text(json.dumps(result, indent=2))
from google.colab import files
files.download("task3-results.json")